In [ ]:
import numpy as np
import pandas as pd

from stochasticbatopt import (
    StorageParams,
    StochasticBatteryOptimizer,
    build_time_index,
    generate_price_scenarios,
)
from stochasticbatopt.plotting import plot_overview

## CONFIG

In [ ]:
# ── date window ───────────────────────────────────────────────────────────
START_DATE = '2027-08-01'
END_DATE   = '2027-08-07'   # exclusive

# ── storage asset ─────────────────────────────────────────────────────────
STORAGE = StorageParams(
    energy_capacity      = 2.0,
    min_energy           = 0.0,
    charge_power         = 1.0,
    discharge_power      = 1.0,
    charge_efficiency    = 0.95,
    discharge_efficiency = 0.95,
    time_step            = 1.0,
)

# ── scenario source ───────────────────────────────────────────────────────
LOAD_CSV      = False
SCENARIOS_CSV = '../da_scenarios.csv'   # used only when LOAD_CSV = True
N_SCENARIOS   = 1000
SIGMA         = 500.0
KAPPA         = 1000.0

# ── optimizer ─────────────────────────────────────────────────────────────
N_LEVELS  = 40
N_BASIS   = 3
CACHE_DIR = '../sbo_cache'
RUN_ID    = 'demo_2027_aug'

## Load / generate scenarios

In [ ]:
time_index = build_time_index(START_DATE, END_DATE)
T          = len(time_index)
print(f'Horizon : {T} hours  ({T // 24} days)')

if LOAD_CSV:
    df             = pd.read_csv(SCENARIOS_CSV, index_col=0)
    scenarios      = df.values
    time_index_csv = pd.DatetimeIndex(df.index, tz='UTC')
    mask           = np.isin(time_index_csv, time_index)
    scenarios      = scenarios[mask, :].T
    time_index     = time_index_csv[mask]
    assert len(time_index) % 24 == 0
    print(f'Scenarios: {scenarios.shape}  (loaded from CSV)')
else:
    scenarios = generate_price_scenarios(
        time_index, SIGMA, KAPPA, N_SCENARIOS, random_state=0
    )
    print(f'Scenarios: {scenarios.shape}  (synthetic OU)')

## Fit model

In [ ]:
model = StochasticBatteryOptimizer(
    params       = STORAGE,
    n_levels     = N_LEVELS,
    n_basis      = N_BASIS,
    cache_dir    = CACHE_DIR,
    run_id       = RUN_ID,
    random_state = 0,
)
result = model.fit(
    price_scenarios = scenarios,
    time_index      = time_index,
    start_energy    = 0.0,
)

## Value decomposition

In [ ]:
vd = model._value_decomp
print(f"Total revenue : {vd['total']:>10.2f} EUR")
print(f"Intrinsic     : {vd['intrinsic']:>10.2f} EUR  ({vd['intr_pct']:.1f}%)")
print(f"Extrinsic     : {vd['extrinsic']:>10.2f} EUR  ({vd['extr_pct']:.1f}%)")
print(f"Std dev       : {result['revenue_per_scenario'].std():>10.2f} EUR")

## Overview plot

In [ ]:
%matplotlib inline
plot_overview(model, result, percentiles=(5, 10), n_plot_days=5)

---
**Save model + result for use in the other notebooks.**

In [ ]:
import pickle, pathlib
pathlib.Path('../sbo_cache').mkdir(exist_ok=True)
with open('../sbo_cache/model_and_result.pkl', 'wb') as fh:
    pickle.dump({'model': model, 'result': result, 'scenarios': scenarios,
                 'time_index': time_index}, fh)
print('Saved to ../sbo_cache/model_and_result.pkl')